# Análise de Dados com Linguagem Python e Apache Spark
## Extraindo e Analisando Dados de Grandes Volumes de Dados

In [1]:
# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [2]:
# Instala o 'kagglehub', a biblioteca oficial do Kaggle para Python.
!pip install kagglehub

In [3]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "André Luiz Colombo" 

Author: André Luiz Colombo



In [4]:
import os
import shutil
import kagglehub

# 1. Definir o diretório de destino
destino_final = "/opt/spark/resources/E-Commerce_Behavior/base"
os.makedirs(destino_final, exist_ok=True)

print("Iniciando o download do dataset do Kaggle...")
# O kagglehub sempre baixa o pacote completo para o cache temporário
path_temporario = kagglehub.dataset_download("egirizkiyansyah44/ecommerce-behavior-data-from-multi-category-store")

# 2. Definir especificamente o arquivo da segunda base (Novembro)
arquivo_desejado = "2019-Nov.csv"
caminho_origem = os.path.join(path_temporario, arquivo_desejado)
caminho_destino = os.path.join(destino_final, arquivo_desejado)

print(f"Download concluído no cache. Movendo apenas a segunda base ({arquivo_desejado})...")

# 3. Move apenas o arquivo de Novembro se ele existir no cache
if os.path.exists(caminho_origem):
    # Se o arquivo já existir no destino, remove antes de mover para não dar erro
    if os.path.exists(caminho_destino):
        os.remove(caminho_destino)
    
    shutil.move(caminho_origem, caminho_destino)
    print(f"Sucesso! O arquivo {arquivo_desejado} foi movido para: {destino_final}")
else:
    print(f"Erro: O arquivo {arquivo_desejado} não foi encontrado no cache.")

print("Processo concluído!")

Iniciando o download do dataset do Kaggle...


/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Download concluído no cache. Movendo apenas a segunda base (2019-Nov.csv)...
Erro: O arquivo 2019-Nov.csv não foi encontrado no cache.
Processo concluído!


In [5]:
# INICIALIZAÇÃO DO ECOSSISTEMA SPARK (SPARK SESSION)

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

# import pyspark.sql.functions import *

spark = (
    SparkSession.builder
        .appName("E-Commerce")
        .master("local[*]")   
        .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-06T21:40:47,442 [Thread-3] WARN  org.apache.hadoop.util.NativeCodeLoader [] - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
# DEFINIÇÃO DE SCHEMA ESTÁTICO E LEITURA DA CAMADA BRONZE (CSV)

schema = T.StructType([
    T.StructField("event_time", T.TimestampType(), True),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("product_id", T.LongType(), True),
    T.StructField("category_id", T.LongType(), True),
    T.StructField("category_code", T.StringType(), True),
    T.StructField("brand", T.StringType(), True),
    T.StructField("price", T.FloatType(), True),
    T.StructField("user_id", T.LongType(), True),
    T.StructField("user_session", T.StringType(), True)
])

# Inicialização da leitura do arquivo CSV bruto
base_df = (spark.read
           .format("csv")
           .option("header", "true")
           .option("sep", ",")
           .schema(schema)
           .load("/opt/spark/resources/E-Commerce_Behavior/base/2019-Oct.csv"))


base_df.show()

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|2053013555631882655|electr

In [7]:
# VERIFICAÇÃO DA ESTRUTURA DOS DADOS (SCHEMA)

base_df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: float (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



In [8]:
#SALVANDO OS DADOS EM FORMATO PARQUET (MIGRAÇÃO PARA A CAMADA SILVER)

(base_df.write.format("parquet")
    .mode("overwrite")
    .save("/opt/spark/resources/E-Commerce_Behavior/base/E-Commerce_parquet")
)

2026-06-06T21:40:58,058 [Executor task launch worker for task 4.0 in stage 1.0 (TID 5)] WARN  org.apache.parquet.hadoop.MemoryManager [] - Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
2026-06-06T21:40:58,060 [Executor task launch worker for task 5.0 in stage 1.0 (TID 6)] WARN  org.apache.parquet.hadoop.MemoryManager [] - Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
2026-06-06T21:40:58,062 [Executor task launch worker for task 9.0 in stage 1.0 (TID 10)] WARN  org.apache.parquet.hadoop.MemoryManager [] - Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
2026-06-06T21:40:58,066 [Executor task launch worker for task 2.0 in stage 1.0 (TID 3)] WARN  org.apache.parquet.hadoop.MemoryManager [] - Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group size

# CAMADA GOLD: AGREGAÇÃO E CONSUMO (BI / ANALYTICS)

<p>A partir desta célula começam as consultas analíticas e agregações de negócio.</p>
<p>O objetivo desta camada é transformar os dados limpos em métricas prontas para
tomada de decisão, relatórios gerenciais e alimentação de dashboards.</p>

In [9]:
# CRIAÇÃO DA TABELA TEMPORÁRIA PARA CONSULTAS SQL

# Cria uma "View" (tabela temporária) na memória do Spark chamada "ecommerce_events".
# Isso permite fazer consultas usando SQL tradicional (spark.sql)

base_df.createOrReplaceTempView("ecommerce_events")

In [10]:
# EVENTOS DE E-COMMERCE: CONSULTA DE TESTE (SQL)

spark.sql('''
          SELECT * FROM ecommerce_events
          LIMIT 10 
          ''').show()

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|2053013555631882655|electr

In [11]:
# EVENTOS DE E-COMMERCE: CONSULTA DE TESTE (DATAFRAME API)

base_df.limit(10).show()

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|2053013555631882655|electr

In [12]:
# ANÁLISE DE FUNIL DE CONVERSÃO (SPARK SQL)

funil_Convercao = spark.sql("""
    SELECT 
        event_type,
        COUNT(*) AS total_eventos,
        COUNT(DISTINCT user_id) AS usuarios_unicos
    FROM ecommerce_events
    GROUP BY event_type
    ORDER BY total_eventos DESC
""")

funil_Convercao.show()

+----------+-------------+---------------+
|event_type|total_eventos|usuarios_unicos|
+----------+-------------+---------------+
|      view|      1016239|         168868|
|  purchase|        17296|          12774|
|      cart|        15040|           7880|
+----------+-------------+---------------+



In [13]:
# ANÁLISE DE FUNIL DE CONVERSÃO (DATAFRAME API)

(base_df
    .groupBy("event_type")
    .agg(
        F.count("*").alias("total_eventos"),
        F.count_distinct("user_id").alias("usuarios_unicos")
    )
    .orderBy(F.col("total_eventos").desc())
).show()

+----------+-------------+---------------+
|event_type|total_eventos|usuarios_unicos|
+----------+-------------+---------------+
|      view|      1016239|         168868|
|  purchase|        17296|          12774|
|      cart|        15040|           7880|
+----------+-------------+---------------+



In [14]:
# ANÁLISE DE DESEMPENHO FINANCEIRO POR MARCA (SPARK SQL)

desempenho_financeiro = spark.sql("""
    SELECT 
        brand,
        COUNT(*) AS quantidade_vendida,
        ROUND(SUM(price), 2) AS faturamento_total,
        ROUND(AVG(price), 2) AS ticket_medio
    FROM ecommerce_events
    WHERE event_type = 'purchase' AND brand IS NOT NULL
    GROUP BY brand
    ORDER BY faturamento_total DESC
    LIMIT 10
""")

desempenho_financeiro.show()

+-------+------------------+-----------------+------------+
|  brand|quantidade_vendida|faturamento_total|ticket_medio|
+-------+------------------+-----------------+------------+
|  apple|              3565|        2780997.8|      780.08|
|samsung|              4053|       1125969.89|      277.81|
| xiaomi|              1337|        206639.65|      154.55|
| huawei|               617|        136513.97|      221.25|
|lucente|               307|         86585.09|      282.04|
|   acer|               159|         84970.77|      534.41|
|     lg|               204|         78768.21|      386.12|
| lenovo|               120|         45590.64|      379.92|
|   oppo|               185|          38619.6|      208.75|
|     hp|               120|         37607.01|      313.39|
+-------+------------------+-----------------+------------+



In [15]:
# ANÁLISE DE DESEMPENHO FINANCEIRO POR MARCA (DATAFRAME API)

(base_df
    .filter((F.col("event_type") == "purchase") & (F.col("brand").isNotNull()))
    .groupBy(F.col("brand"))
    .agg(
        F.count(F.col("*")).alias("quantidade_vendida"),
        F.round(F.sum(F.col("price")), 2).alias("faturamento_total"),
        F.round(F.avg(F.col("price")), 2).alias("ticket_medio")
    )
    .orderBy(F.col("faturamento_total").desc())
    .limit(10)
).show()

+-------+------------------+-----------------+------------+
|  brand|quantidade_vendida|faturamento_total|ticket_medio|
+-------+------------------+-----------------+------------+
|  apple|              3565|        2780997.8|      780.08|
|samsung|              4053|       1125969.89|      277.81|
| xiaomi|              1337|        206639.65|      154.55|
| huawei|               617|        136513.97|      221.25|
|lucente|               307|         86585.09|      282.04|
|   acer|               159|         84970.77|      534.41|
|     lg|               204|         78768.21|      386.12|
| lenovo|               120|         45590.64|      379.92|
|   oppo|               185|          38619.6|      208.75|
|     hp|               120|         37607.01|      313.39|
+-------+------------------+-----------------+------------+



In [16]:
# ANÁLISE DE COMPORTAMENTO DO USUÁRIO POR HORA DO DIA (SPARK SQL)

comportamento_hora = spark.sql("""
    SELECT 
        HOUR(event_time) AS hora_do_dia,
        COUNT(CASE WHEN event_type = 'view' THEN 1 END) AS total_visualizacoes,
        COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS total_compras
    FROM ecommerce_events
    GROUP BY hora_do_dia
    ORDER BY hora_do_dia ASC
""")

comportamento_hora.show()

+-----------+-------------------+-------------+
|hora_do_dia|total_visualizacoes|total_compras|
+-----------+-------------------+-------------+
|          0|               1070|           10|
|          1|                121|            0|
|          2|              22326|          316|
|          3|              47951|          845|
|          4|              53390|         1021|
|          5|              61234|         1276|
|          6|              63713|         1393|
|          7|              68550|         1409|
|          8|              70767|         1464|
|          9|              68482|         1478|
|         10|              63570|         1339|
|         11|              62162|         1204|
|         12|              60799|         1120|
|         13|              64772|         1055|
|         14|              76990|         1017|
|         15|              91425|          985|
|         16|              97348|          961|
|         17|              41569|       

In [17]:
# ANÁLISE DE COMPORTAMENTO DO USUÁRIO POR HORA DO DIA (DATAFRAME API)

(base_df
    .withColumn("hora_do_dia", F.hour(F.col("event_time")))
    .groupBy("hora_do_dia")
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("total_visualizacoes"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("total_compras")
    )
    .orderBy(F.col("hora_do_dia").asc())
).show()

+-----------+-------------------+-------------+
|hora_do_dia|total_visualizacoes|total_compras|
+-----------+-------------------+-------------+
|          0|               1070|           10|
|          1|                121|            0|
|          2|              22326|          316|
|          3|              47951|          845|
|          4|              53390|         1021|
|          5|              61234|         1276|
|          6|              63713|         1393|
|          7|              68550|         1409|
|          8|              70767|         1464|
|          9|              68482|         1478|
|         10|              63570|         1339|
|         11|              62162|         1204|
|         12|              60799|         1120|
|         13|              64772|         1055|
|         14|              76990|         1017|
|         15|              91425|          985|
|         16|              97348|          961|
|         17|              41569|       

In [18]:
# ANÁLISE DE CATEGORIAS: DESEJADAS VS. COMPRADAS (SPARK SQL)

desejadas_vs_compradas = spark.sql("""
    SELECT 
        category_code,
        COUNT(CASE WHEN event_type = 'view' THEN 1 END) AS visualizacoes,
        COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS compras,
        ROUND((COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) * 100.0 / COUNT(CASE WHEN event_type = 'view' THEN 1 END)), 2) AS taxa_conversao_pct
    FROM ecommerce_events
    WHERE category_code IS NOT NULL
    GROUP BY category_code
    HAVING visualizacoes > 1000  -- Filtra categorias irrelevantes
    ORDER BY compras DESC
    LIMIT 10
""")

desejadas_vs_compradas.show()

+--------------------+-------------+-------+------------------+
|       category_code|visualizacoes|compras|taxa_conversao_pct|
+--------------------+-------------+-------+------------------+
|electronics.smart...|       268272|   8110|              3.02|
|electronics.audio...|        25677|    798|              3.11|
|electronics.video.tv|        22320|    460|              2.06|
|  electronics.clocks|        36473|    434|              1.19|
|  computers.notebook|        29642|    416|              1.40|
|appliances.kitche...|        16410|    307|              1.87|
|appliances.enviro...|        17706|    284|              1.60|
|appliances.kitche...|        18528|    232|              1.25|
|appliances.enviro...|        10202|    142|              1.39|
|  electronics.tablet|         7386|    115|              1.56|
+--------------------+-------------+-------+------------------+



In [19]:
# ANÁLISE DE CATEGORIAS: DESEJADAS VS. COMPRADAS (DATAFRAME API)

(base_df
    .filter(F.col("category_code").isNotNull())
    .groupBy("category_code")
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("visualizacoes"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("compras")
    )
    .filter(F.col("visualizacoes") > 1000)
    .withColumn(
        "taxa_conversao_pct",
        F.round((F.col("compras") * 100.0) / F.col("visualizacoes"), 2)
    )
    .orderBy(F.col("compras").desc())
    .limit(10)
).show()

+--------------------+-------------+-------+------------------+
|       category_code|visualizacoes|compras|taxa_conversao_pct|
+--------------------+-------------+-------+------------------+
|electronics.smart...|       268272|   8110|              3.02|
|electronics.audio...|        25677|    798|              3.11|
|electronics.video.tv|        22320|    460|              2.06|
|  electronics.clocks|        36473|    434|              1.19|
|  computers.notebook|        29642|    416|               1.4|
|appliances.kitche...|        16410|    307|              1.87|
|appliances.enviro...|        17706|    284|               1.6|
|appliances.kitche...|        18528|    232|              1.25|
|appliances.enviro...|        10202|    142|              1.39|
|  electronics.tablet|         7386|    115|              1.56|
+--------------------+-------------+-------+------------------+



In [20]:
# ANÁLISE DE CLIENTES VIP: MAIORES COMPRADORES (SPARK SQL)

clientes_vip = spark.sql("""
    SELECT 
        user_id,
        COUNT(*) AS total_itens_comprados,
        ROUND(SUM(price), 2) AS total_gasto
    FROM ecommerce_events
    WHERE event_type = 'purchase'
    GROUP BY user_id
    ORDER BY total_gasto DESC
    LIMIT 10
""")

clientes_vip.show()

+---------+---------------------+-----------+
|  user_id|total_itens_comprados|total_gasto|
+---------+---------------------+-----------+
|541337272|                   17|    15742.3|
|519058946|                   10|   15092.98|
|542561762|                    7|    12404.4|
|554090147|                   26|   12066.53|
|531974857|                   17|    11540.5|
|520695559|                   16|   11047.08|
|555206577|                    5|    9665.62|
|553125023|                    5|    9356.73|
|552265052|                   12|    8854.15|
|515994290|                    8|    8486.21|
+---------+---------------------+-----------+



In [21]:
# ANÁLISE DE CLIENTES VIP: MAIORES COMPRADORES (DATAFRAME API)

(base_df
    .filter(F.col("event_type") == "purchase")
    .groupBy("user_id")
    .agg(
        F.count("*").alias("total_itens_comprados"),
        F.round(F.sum("price"), 2).alias("total_gasto")
    )
    .orderBy(F.col("total_gasto").desc())
    .limit(10)
).show()

+---------+---------------------+-----------+
|  user_id|total_itens_comprados|total_gasto|
+---------+---------------------+-----------+
|541337272|                   17|    15742.3|
|519058946|                   10|   15092.98|
|542561762|                    7|    12404.4|
|554090147|                   26|   12066.53|
|531974857|                   17|    11540.5|
|520695559|                   16|   11047.08|
|555206577|                    5|    9665.62|
|553125023|                    5|    9356.73|
|552265052|                   12|    8854.15|
|515994290|                    8|    8486.21|
+---------+---------------------+-----------+



In [22]:
# Finaliza a SparkSession atual e libera todos os recursos alocados.
# Essa etapa é fundamental para liberar a memória RAM e os núcleos de processamento 
# da máquina (ou do cluster) que estavam reservados para o Spark, evitando 
# vazamento de memória (memory leak) após a conclusão do pipeline.

spark.stop()

In [23]:
%reload_ext watermark
%watermark -a "André Luiz Colombo"

Author: André Luiz Colombo



In [24]:
%watermark -v -m

Python implementation: CPython
Python version       : 3.8.10
IPython version      : 8.12.3

Compiler    : GCC 9.4.0
OS          : Linux
Release     : 6.12.76-linuxkit
Machine     : x86_64
Processor   : x86_64
CPU cores   : 12
Architecture: 64bit



In [25]:
%watermark --iversions

kagglehub: 0.2.9
pyspark  : 3.5.1



# Fim